In [ ]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser
from sklearn.metrics.pairwise import cosine_similarity
import re


df_clean = pd.read_pickle('../data/cleaned_all_recipes.pkl')
print(f"โหลดข้อมูลสำเร็จ! จำนวนสูตรอาหารทั้งหมด: {len(df_clean)} เมนู")

โหลดข้อมูลสำเร็จ! จำนวนสูตรอาหารทั้งหมด: 231636 เมนู


In [ ]:
df_clean.head()

,name,id,minutes,contributor_id,submitted,tags,n_steps,steps,description,ingredients,n_ingredients,calories,total_fat,sugar,sodium,protein,sat_fat,carbs,category
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"[60-minutes-or-less, time-to-make, course, mai...",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7,51.5,0.0,13.0,0.0,2.0,0.0,4.0,Dessert
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"[30-minutes-or-less, time-to-make, course, mai...",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6,173.4,18.0,0.0,17.0,22.0,35.0,1.0,Dessert
2,all in the kitchen chili,112140,130,196586,2005-02-25,"[time-to-make, course, preparation, main-dish,...",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13,269.8,22.0,32.0,48.0,39.0,27.0,5.0,Soup/Curry
3,alouette potatoes,59389,45,68585,2003-04-14,"[60-minutes-or-less, time-to-make, course, mai...",11,[place potatoes in a large pot of lightly salt...,"this is a super easy, great tasting, make ahea...","[spreadable cheese with garlic and herbs, new ...",11,368.1,17.0,10.0,2.0,14.0,8.0,20.0,Breakfast
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"[weeknight, time-to-make, course, main-ingredi...",5,"[mix all ingredients& boil for 2 1 / 2 hours ,...",my dh's amish mother raised him on this recipe...,"[tomato juice, apple cider vinegar, sugar, sal...",8,352.9,1.0,337.0,23.0,3.0,0.0,28.0,Main Course


In [ ]:
def tokenize_ingredients(ingredient_list):
    
    text = " ".join(ingredient_list).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    
    return text.split() 


df_clean['tokenized_ingredients'] = df_clean['ingredients'].apply(tokenize_ingredients)

print("ตัวอย่างข้อมูลที่ตัดคำแล้ว (แต่ยังไม่รวมคำคู่):")
print(df_clean['tokenized_ingredients'].head(3).values)

ตัวอย่างข้อมูลที่ตัดคำแล้ว (แต่ยังไม่รวมคำคู่):
[list(['winter', 'squash', 'mexican', 'seasoning', 'mixed', 'spice', 'honey', 'butter', 'olive', 'oil', 'salt'])
 list(['prepared', 'pizza', 'crust', 'sausage', 'patty', 'eggs', 'milk', 'salt', 'and', 'pepper', 'cheese'])
 list(['ground', 'beef', 'yellow', 'onions', 'diced', 'tomatoes', 'tomato', 'paste', 'tomato', 'soup', 'rotel', 'tomatoes', 'kidney', 'beans', 'water', 'chili', 'powder', 'ground', 'cumin', 'salt', 'lettuce', 'cheddar', 'cheese'])]


In [ ]:
print("กำลังค้นหาคำที่มักใช้คู่กัน (Bigrams)...")
raw_sentences = df_clean['tokenized_ingredients'].tolist()


phrases = Phrases(raw_sentences, min_count=3, threshold=10) 
bigram_model = Phraser(phrases)


df_clean['tokenized_ingredients'] = [bigram_model[sent] for sent in raw_sentences]

print("ตัวอย่างข้อมูลที่รวมคำ (Phrases) เป็น Bigram แล้ว:")
print(df_clean['tokenized_ingredients'].head(3).values)

กำลังค้นหาคำที่มักใช้คู่กัน (Bigrams)...
ตัวอย่างข้อมูลที่รวมคำ (Phrases) เป็น Bigram แล้ว:
[list(['winter_squash', 'mexican', 'seasoning', 'mixed', 'spice', 'honey', 'butter', 'olive', 'oil', 'salt'])
 list(['prepared', 'pizza_crust', 'sausage', 'patty', 'eggs', 'milk', 'salt', 'and', 'pepper', 'cheese'])
 list(['ground', 'beef', 'yellow', 'onions', 'diced', 'tomatoes', 'tomato', 'paste', 'tomato', 'soup', 'rotel', 'tomatoes', 'kidney_beans', 'water', 'chili', 'powder', 'ground', 'cumin', 'salt', 'lettuce', 'cheddar', 'cheese'])]


In [ ]:
sentences = df_clean['tokenized_ingredients'].tolist()

print("กำลังเทรน Word2Vec Model... (อาจใช้เวลาสักครู่)")
word2vec_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4)
print("เทรนสำเร็จ!")


print("\nคำที่คล้ายกับ 'lemon' ที่สุดในมุมมองของ AI:")
for word, score in word2vec_model.wv.most_similar('lemon', topn=5):
    print(f" - {word}: ความเหมือน {score:.4f}")

กำลังเทรน Word2Vec Model... (อาจใช้เวลาสักครู่)
เทรนสำเร็จ!

คำที่คล้ายกับ 'lemon' ที่สุดในมุมมองของ AI:
 - orange: ความเหมือน 0.7849
 - lemons: ความเหมือน 0.7826
 - lime: ความเหมือน 0.7762
 - key_lime: ความเหมือน 0.5810
 - tangerine: ความเหมือน 0.5676


In [ ]:
def get_recipe_vector(tokens, model):
    
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    return np.mean(vectors, axis=0)

print("กำลังแปลงสูตรอาหารทั้งหมดให้เป็น Vector Matrix...")


recipe_vectors = np.array([get_recipe_vector(tokens, word2vec_model) for tokens in df_clean['tokenized_ingredients']])

print(f"ขนาดของ Recipe Matrix: {recipe_vectors.shape}")

กำลังแปลงสูตรอาหารทั้งหมดให้เป็น Vector Matrix...
ขนาดของ Recipe Matrix: (231636, 100)


In [ ]:
def search_fridge_word2vec(user_input, top_k=5):
    print(f"🧺 ของในตู้เย็น: '{user_input}'")
    
    raw_user_tokens = tokenize_ingredients([user_input])

    user_tokens = bigram_model[raw_user_tokens] 
    print(f"🔍 คำที่ AI มองเห็น: {user_tokens}")

    user_vec = get_recipe_vector(user_tokens, word2vec_model).reshape(1, -1)
    
    if np.all(user_vec == 0):
        print("AI ไม่รู้จักวัตถุดิบเหล่านี้เลย ลองพิมพ์ใหม่นะครับ")
        return None
    
    similarities = cosine_similarity(user_vec, recipe_vectors)[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = df_clean.iloc[top_indices][['name', 'category', 'ingredients']].copy()
    results['match_score'] = similarities[top_indices].round(4)
    
    return results

# ==========================================
# 🎯 ลองทดสอบระบบ
# ==========================================
result_df = search_fridge_word2vec("chicken lime garlic powder", top_k=5)
display(result_df)

🧺 ของในตู้เย็น: 'chicken lime garlic powder'
🔍 คำที่ AI มองเห็น: ['chicken', 'lime', 'garlic', 'powder']


,name,category,ingredients,match_score
124099,lime and cumin roasted chicken,Baked/Roast,"[chicken, lime, paprika, cumin, salt and pepper]",0.8304
195585,spicy chicken with coconut lime sauce,Pasta/Noodle,"[chicken thighs, chicken legs, peanut oil, gre...",0.8173
115632,juan s buttery garlic chicken,Baked/Roast,"[boneless skinless chicken breasts, butter, ga...",0.8159
87568,fried chicken breast without breading,Fried/Stir-Fry,"[boneless skinless chicken breasts, seasoning ...",0.8089
151035,oven tandoori chicken,Dessert,"[vegetable oil, garlic cloves, fresh ginger, g...",0.8055


In [ ]:
import joblib

print("กำลังบันทึก Model ทั้งหมดลงโฟลเดอร์ models/ ...")

joblib.dump(word2vec_model, '../models/word2vec_model.joblib')

joblib.dump(recipe_vectors, '../models/recipe_vectors.joblib')

joblib.dump(bigram_model, '../models/bigram_model.joblib')

df_for_api = df_clean[['name', 'category', 'ingredients']].copy()
joblib.dump(df_for_api, '../models/df_for_api.joblib')

print("✅ บันทึกเสร็จสมบูรณ์! ตอนนี้คุณพร้อมทำ API แล้วครับ")

กำลังบันทึก Model ทั้งหมดลงโฟลเดอร์ models/ ...
✅ บันทึกเสร็จสมบูรณ์! ตอนนี้คุณพร้อมทำ API แล้วครับ


In [ ]:
print(f"จำนวนเมนูทั้งหมด: {len(df_clean)} เมนู")

จำนวนเมนูทั้งหมด: 231636 เมนู
